# schedule

Makes a Scheduled run, a notebook an outside scheduler runs top to bottom, from admin/templates/scheduled_run.py: you fill in its cells in its own notebook, in scheduled_runs/.

## Terms

- **Step**: each numbered heading below, with the code cell under it. Its first line says its kind: it changes only files, or only prints; it is a call to the warehouse, yours to run; or it is a decision, yours to make.
- **Names cell**: the code cell after the one that finds the Project. It holds this Workflow's inputs, the start materials you fill in once, each with what to write in its comment; each step's call reads them.
- **Scheduled run**: a notebook in scheduled_runs/<name>/ that an outside scheduler, such as papermill, runs top to bottom: it writes the tables it needs, in order, and gives back what you asked for.
- **Statement**: a query your code builds, which becomes one Hive string.
- **Hive**: the SQL your warehouse runs.

## Before step 1

Fill in the names cell below and save the notebook: each call reads it, and so do `project.check()`, which names the next step, and `project.do_next()`, the cell before the last.

## Decision rules

- If you run the steps yourself, run each step's cell in turn, in order.
- If you want them run for you, run the last cell, `project.run_workflow(...)`, in place of their cells. It runs every step left in turn, up to the first only you can do, and says where it stopped: go on from that step.
- If you want only the steps that change files run for you, run `project.do_next()`, the cell before the last. It runs them up to the first it can't.

## Self-check

Once the last step is done, each of these holds:

- scheduled_runs/<name>/<name>.ipynb is there for each name in `NAME`.
- `project.check()` names no next step of this Workflow.

In [ ]:
import pathlib
import sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[2]))  # the Project: three up

from sqlglot_composer import admin, hive_settings, project, run_execute, run_query, show_hive
from warehouse import connect, query, execute

# Your connection to the warehouse. Each notebook runs on its own, so each connects here, at
# its top, before it calls query or execute: write yours in the line below and delete its #,
# such as connect(your_library.connect(...)). A connection kept in a file outside the Project,
# such as team_connection.py, is imported by these lines, written in that line's place:
#     sys.path.append("C:/work/connections")  # the folder team_connection.py is in
#     from team_connection import conn
#     connect(conn)
import example_database
connect(example_database)


In [ ]:
NAME = 'weekly_team'  # the Scheduled run's name, a Python name such as "weekly", or a list of them

## 1. Make the Scheduled run's notebook

It changes only files, so project.do_next() runs it for you.

- **Goal:** the notebook of the Scheduled run `NAME` names, scheduled_runs/<name>/<name>.ipynb, made from admin/templates/scheduled_run.py and registered.
- **Action:** Run the cell below. It refuses a name already taken, or one Python can't import. Once the run's notebook is there, this step is done, and running the cell again is refused, as the name is taken. Then open the run's notebook and fill in its `<...>`: its parameters, its connection and the Statements it runs, as its own text cell says.
- **Done when** scheduled_runs/<name>/<name>.ipynb is there; the Workflow is done: go through the Self-check at the top.

In [ ]:
project.new_scheduled_run(NAME)

In [ ]:
project.do_next()

In [ ]:
# Every step left, in place of their cells, up to the first only you can do.
# Add dry_run=True to see the Hive it would send, sending nothing through your
# execute: help(project.run_workflow) says the steps it still runs:
project.run_workflow("schedule")